# Exploratory Data Analysis
## Credit Card Fraud Detection

**Датасет:** [Credit Card Fraud Detection — Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Мета:** Дослідити структуру та якість даних, виявити патерни шахрайських транзакцій та підготувати ознаки для подальшого навчання моделей машинного навчання.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize']   = 14
plt.rcParams['axes.titleweight'] = 'bold'

---
## 1. Dataset Overview

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.info()

In [ ]:
df.head()

**Висновок:**  
Датасет містить **284 807 транзакцій** та **31 ознаку**:
- `V1`–`V28` — 28 анонімізованих компонентів PCA (конфіденційність);
- `Time` — секунди з моменту першої транзакції;
- `Amount` — сума транзакції;
- `Class` — цільова змінна (0 = нормальна, 1 = шахрайство).

---
## 2. Data Quality Assessment

In [ ]:
missing    = df.isnull().sum()
duplicates = df.duplicated().sum()

print(f"Columns with missing values: {(missing > 0).sum()}")
print(f"Duplicate rows:              {duplicates}")

In [ ]:
quality = pd.DataFrame({
    'Check':  ['Missing values', 'Duplicate rows'],
    'Result': [int(missing.sum()), int(duplicates)]
})
display(quality)

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"Removed {duplicates} duplicate rows. New shape: {df.shape}")

**Висновок:**  
Датасет не містить пропущених значень. Якість даних оцінюється як висока — додаткова імпутація не потрібна.

---
## 3. Target Variable Analysis

In [ ]:
fraud_rate = df['Class'].value_counts(normalize=True) * 100
print(fraud_rate)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='Class', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Class Distribution')
plt.xlabel('Class  (0 = Normal, 1 = Fraud)')
plt.ylabel('Count')
plt.yscale('log')
for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height()):,}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom', fontsize=11,
        xytext=(0, 5), textcoords='offset points'
    )
plt.tight_layout()
plt.show()

In [ ]:
class_counts = df['Class'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(
    class_counts,
    labels=['Normal', 'Fraud'],
    autopct='%1.2f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90,
    explode=(0, 0.07),
    shadow=True
)
plt.title('Class Distribution (Pie)')
plt.show()

**Висновок:**  
Датасет є **сильно незбалансованим**. Частка шахрайських транзакцій становить близько **0.17%**, що потребує використання спеціалізованих метрик оцінки моделей (Precision-Recall AUC, F1-score) та технік балансування вибірки (SMOTE, under-sampling).

---
## 4. Transaction Amount Analysis

In [ ]:
df['Amount'].describe()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Amount'], bins=100, color='#3498db', kde=True)
plt.title('Transaction Amount Distribution (Original)')
plt.xlabel('Amount')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Log-transform: стандартний best practice для цього датасету
df['Amount_log'] = np.log1p(df['Amount'])

plt.figure(figsize=(10, 6))
sns.histplot(df['Amount_log'], bins=100, color='#3498db', kde=True)
plt.title('Transaction Amount Distribution (log1p)')
plt.xlabel('log1p(Amount)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Class', y='Amount_log', data=df,
            palette=['#2ecc71', '#e74c3c'])
plt.title('Amount (log) by Class')
plt.xlabel('Class  (0 = Normal, 1 = Fraud)')
plt.ylabel('log1p(Amount)')
plt.tight_layout()
plt.show()

**Висновок:**  
Розподіл `Amount` є сильно правостороннім (right-skewed). Логарифмічне перетворення `log1p` нормалізує розподіл та зменшує вплив викидів. З boxplot видно, що шахрайські транзакції в середньому мають **меншу суму** — зловмисники часто маскуються під дрібні операції.

---
## 5. Time Analysis

In [ ]:
# Перетворення секунд у години доби — стандартний підхід для цього датасету
df['Hour'] = (df['Time'] / 3600) % 24

plt.figure(figsize=(10, 5))
sns.histplot(df['Hour'], bins=24, color='#9b59b6', kde=True)
plt.title('Transaction Count by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Class', y='Hour', data=df,
            palette=['#2ecc71', '#e74c3c'])
plt.title('Hour of Day by Class')
plt.xlabel('Class  (0 = Normal, 1 = Fraud)')
plt.ylabel('Hour')
plt.tight_layout()
plt.show()

**Висновок:**  
Нормальні транзакції демонструють чіткий добовий патерн із піком активності вдень та спадом вночі. Шахрайські транзакції розподілені більш рівномірно впродовж доби, що відображено у ширшому boxplot для класу 1.

---
## 6. Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

target_corr = corr['Class'].drop('Class').sort_values(ascending=False)
print("Top correlations with Class:")
print(target_corr.head(15))

In [ ]:
top_features = target_corr.abs().sort_values(ascending=False)[1:11]

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if target_corr[f] > 0 else '#3498db' for f in top_features.index]
sns.barplot(x=top_features.values, y=top_features.index, palette=colors)
plt.title('Top-10 Features Correlated with Class (|Pearson|)')
plt.xlabel('|Correlation|')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

**Висновок:**  
Ознаки `V14`, `V12`, `V10`, `V16`, `V11` демонструють найвищий рівень кореляції з цільовою змінною. Ці ознаки стануть пріоритетними при аналізі Feature Importance на етапі моделювання.

---
## 7. Feature Distribution Analysis

In [ ]:
top6 = ['V14', 'V12', 'V10', 'V17', 'V16', 'V11']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top6):
    sns.kdeplot(
        df[df['Class'] == 0][feat], ax=axes[i],
        label='Normal', color='#2ecc71', fill=True, alpha=0.4
    )
    sns.kdeplot(
        df[df['Class'] == 1][feat], ax=axes[i],
        label='Fraud', color='#e74c3c', fill=True, alpha=0.4
    )
    axes[i].set_title(f'Distribution: {feat}')
    axes[i].set_xlabel(feat)
    axes[i].legend(fontsize=9)

plt.suptitle('KDE: Top-6 PCA Features — Normal vs Fraud',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Висновок:**  
Усі шість ознак демонструють виразне розмежування між класами. `V14`, `V12` та `V10` мають мінімальне перекриття розподілів — це свідчить про їхню **високу дискримінативну силу** та пояснює їхню позицію у топі кореляцій.

---
## 8. PCA Feature Relationships

In [ ]:
# Підвибірка нормальних транзакцій для читабельності графіка
normal_sample = df[df['Class'] == 0].sample(3000, random_state=42)
fraud_all     = df[df['Class'] == 1]
plot_df = pd.concat([normal_sample, fraud_all])

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='V14', y='V12',
    hue='Class',
    data=plot_df,
    palette={0: '#2ecc71', 1: '#e74c3c'},
    alpha=0.55,
    s=18,
    edgecolors='none'
)
plt.title('Feature Space: V14 vs V12  (Normal vs Fraud)')
plt.xlabel('V14')
plt.ylabel('V12')
handles, _ = plt.gca().get_legend_handles_labels()
plt.legend(handles, ['Normal', 'Fraud'])
plt.tight_layout()
plt.show()

**Висновок:**  
Scatter-графік наочно підтверджує, що шахрайські транзакції займають **специфічні регіони** у просторі PCA-ознак та утворюють відокремлені кластери. Це обґрунтовує застосування як лінійних класифікаторів, так і нелінійних ансамблевих методів.

---
## 9. EDA Summary

### Key Findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Dataset contains **severe class imbalance** (~0.17% fraud) | Use PR-AUC, F1; apply SMOTE / under-sampling |
| 2 | **Dataset quality is high** — no missing values | No imputation required |
| 3 | Amount distribution is **highly right-skewed** | `log1p` transformation required |
| 4 | Fraudulent amounts tend to be **lower** on average | `Amount_log` carries predictive signal |
| 5 | Fraudulent transactions are **more uniform across hours** | `Hour` feature adds temporal signal |
| 6 | `V14`, `V12`, `V10`, `V17`, `V16`, `V11` show **strongest class separation** | Expected to rank high in Feature Importance |
| 7 | Fraud transactions form **distinct clusters** in PCA space | Supports both linear and non-linear classifiers |

---

**Наступний етап →** `02_Model_Training.ipynb`